In [1]:
import pandas as pd
import zipfile
import os

print("pandas version:", pd.__version__)
print("ready")

pandas version: 2.2.2
ready


In [4]:
from google.colab import files
uploaded = files.upload()

print("\nFiles now in Colab:")
for name in uploaded:
    print(" -", name)

Saving 2021_GCP_POA_for_QLD_short-header.zip to 2021_GCP_POA_for_QLD_short-header.zip

Files now in Colab:
 - 2021_GCP_POA_for_QLD_short-header.zip


In [5]:
import os
for f in os.listdir():
    if f.endswith('.xlsx') or f.endswith('.zip'):
        size = os.path.getsize(f) / (1024*1024)
        print(f"{f}  ({size:.1f} MB)")

2021_GCP_POA_for_QLD_short-header.zip  (9.8 MB)
rta-bond-statistics.xlsx  (5.8 MB)
2016_GCP_POA_for_QLD_short-header.zip  (9.2 MB)


In [6]:
raw = pd.read_excel('rta-bond-statistics.xlsx', sheet_name='1 pc-rents', header=None)

print("Shape (rows, columns):", raw.shape)
print("\nFirst 10 rows, first 6 columns:")
print(raw.iloc[0:10, 0:6])

Shape (rows, columns): (2560, 61)

First 10 rows, first 6 columns:
    0                                                  1         2         3  \
0 NaN                                           Contents       NaN       NaN   
1 NaN                          Median rents, by Postcode       NaN       NaN   
2 NaN  Median value of weekly rent paid for new tenan...       NaN       NaN   
3 NaN                                                NaN       NaN       NaN   
4 NaN                                                NaN  Postcode  Dwelling   
5 NaN                                                NaN       NaN       NaN   
6 NaN                                                NaN       NaN       NaN   
7 NaN                                                NaN       NaN       NaN   
8 NaN                                                NaN      4000    Flat 1   
9 NaN                                                NaN      4000    Flat 2   

      4     5  
0   NaN   NaN  
1   NaN   NaN  
2   

In [7]:
quarter_row = raw.iloc[5]   # row with Mar/Jun/Sep/Dec
year_row = raw.iloc[6]      # row with the years

for col in range(raw.shape[1]):
    q = quarter_row[col]
    y = year_row[col]
    if q == 'Sep' and y in (2016, 2021):
        print(f"Column {col}: {q} {y}")

Column 22: Sep 2016
Column 42: Sep 2021


In [8]:
data = raw.iloc[8:, [2, 3, 22, 42]].copy()
data.columns = ['postcode', 'dwelling', 'rent_2016', 'rent_2021']

# drop rows where postcode is blank (trailing empty rows, separators)
data = data.dropna(subset=['postcode'])

print("Shape:", data.shape)
print(data.head(10))

Shape: (2552, 4)
   postcode     dwelling rent_2016 rent_2021
8      4000       Flat 1       400       380
9      4000       Flat 2       550       500
10     4000       Flat 3       750       670
11     4000      House 2       425       515
12     4000      House 3       597       590
13     4000      House 4       850       642
14     4000  Townhouse 2       535       NaN
15     4000  Townhouse 3       610       NaN
16     4005       Flat 1       380       400
17     4005       Flat 2       492       542


In [9]:
flat2 = data[data['dwelling'] == 'Flat 2'].copy()

# make sure rents are numeric
flat2['rent_2016'] = pd.to_numeric(flat2['rent_2016'], errors='coerce')
flat2['rent_2021'] = pd.to_numeric(flat2['rent_2021'], errors='coerce')

# keep only postcodes with a value in BOTH years
flat2 = flat2.dropna(subset=['rent_2016', 'rent_2021'])

print("Flat 2 postcodes with both years:", flat2.shape[0])
print(flat2.head(10))

Flat 2 postcodes with both years: 143
   postcode dwelling  rent_2016  rent_2021
9      4000   Flat 2      550.0      500.0
17     4005   Flat 2      492.0      542.0
25     4006   Flat 2      490.0      500.0
33     4007   Flat 2      430.0      485.0
41     4010   Flat 2      450.0      450.0
49     4011   Flat 2      340.0      370.0
57     4012   Flat 2      375.0      400.0
65     4013   Flat 2      320.0      307.0
81     4017   Flat 2      305.0      320.0
97     4019   Flat 2      275.0      330.0


In [10]:
flat2['rent_change_abs'] = flat2['rent_2021'] - flat2['rent_2016']
flat2['rent_change_pct'] = round((flat2['rent_2021'] - flat2['rent_2016']) / flat2['rent_2016'] * 100, 1)

print(flat2[['postcode', 'rent_2016', 'rent_2021', 'rent_change_abs', 'rent_change_pct']].head(10))
print("\nMedian rent change pct:", flat2['rent_change_pct'].median())
print("Postcodes where rent rose:", (flat2['rent_change_abs'] > 0).sum())
print("Postcodes where rent fell:", (flat2['rent_change_abs'] < 0).sum())

   postcode  rent_2016  rent_2021  rent_change_abs  rent_change_pct
9      4000      550.0      500.0            -50.0             -9.1
17     4005      492.0      542.0             50.0             10.2
25     4006      490.0      500.0             10.0              2.0
33     4007      430.0      485.0             55.0             12.8
41     4010      450.0      450.0              0.0              0.0
49     4011      340.0      370.0             30.0              8.8
57     4012      375.0      400.0             25.0              6.7
65     4013      320.0      307.0            -13.0             -4.1
81     4017      305.0      320.0             15.0              4.9
97     4019      275.0      330.0             55.0             20.0

Median rent change pct: 13.9
Postcodes where rent rose: 130
Postcodes where rent fell: 9


In [11]:
flat2.to_csv('rents_flat2_clean.csv', index=False)

from google.colab import files
files.download('rents_flat2_clean.csv')

print("Saved and downloading rents_flat2_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloading rents_flat2_clean.csv


In [13]:
import zipfile, os

with zipfile.ZipFile('2016_GCP_POA_for_QLD_short-header.zip', 'r') as z:
    z.extractall('abs2016')

# find the G02 file for postal areas (POA)
for folder, subfolders, filenames in os.walk('abs2016'):
    for name in filenames:
        if 'G02' in name and 'POA' in name:
            print(os.path.join(folder, name))

abs2016/2016 Census GCP Postal Areas for QLD/2016Census_G02_QLD_POA.csv


In [14]:
inc2016 = pd.read_csv(
    'abs2016/2016 Census GCP Postal Areas for QLD/2016Census_G02_QLD_POA.csv',
    dtype={'POA_CODE_2016': str}
)

print("All columns:", list(inc2016.columns))
print("\nShape:", inc2016.shape)

# keep just postcode code and median household income
inc2016 = inc2016[['POA_CODE_2016', 'Median_tot_hhd_inc_weekly']]
print(inc2016.head())

All columns: ['POA_CODE_2016', 'Median_age_persons', 'Median_mortgage_repay_monthly', 'Median_tot_prsnl_inc_weekly', 'Median_rent_weekly', 'Median_tot_fam_inc_weekly', 'Average_num_psns_per_bedroom', 'Median_tot_hhd_inc_weekly', 'Average_household_size']

Shape: (442, 9)
  POA_CODE_2016  Median_tot_hhd_inc_weekly
0       POA4000                       1728
1       POA4005                       2054
2       POA4006                       1618
3       POA4007                       1918
4       POA4008                       1199


In [15]:
inc2016 = inc2016.copy()
inc2016['postcode'] = inc2016['POA_CODE_2016'].str.replace('POA', '', regex=False)
inc2016 = inc2016[['postcode', 'Median_tot_hhd_inc_weekly']]
inc2016.columns = ['postcode', 'income_2016']

print(inc2016.head())

  postcode  income_2016
0     4000         1728
1     4005         2054
2     4006         1618
3     4007         1918
4     4008         1199


In [16]:
# 1. unzip the 2021 file
with zipfile.ZipFile('2021_GCP_POA_for_QLD_short-header.zip', 'r') as z:
    z.extractall('abs2021')

# 2. load the G02 file for 2021
inc2021 = pd.read_csv(
    'abs2021/2021 Census GCP Postal Areas for QLD/2021Census_G02_QLD_POA.csv',
    dtype={'POA_CODE_2021': str}
)

# 3. strip the POA prefix, keep the two columns, rename
inc2021['postcode'] = inc2021['POA_CODE_2021'].str.replace('POA', '', regex=False)
inc2021 = inc2021[['postcode', 'Median_tot_hhd_inc_weekly']]
inc2021.columns = ['postcode', 'income_2021']

print(inc2021.head())

  postcode  income_2021
0     4000         1860
1     4005         2258
2     4006         1871
3     4007         2130
4     4008         1666


In [17]:
# join rents + 2016 income
merged = flat2.merge(inc2016, on='postcode', how='inner')

# then join on 2021 income
merged = merged.merge(inc2021, on='postcode', how='inner')

print("Postcodes after join:", merged.shape[0])
print(merged[['postcode', 'rent_2016', 'rent_2021', 'income_2016', 'income_2021']].head(10))

Postcodes after join: 0
Empty DataFrame
Columns: [postcode, rent_2016, rent_2021, income_2016, income_2021]
Index: []


In [18]:
print("flat2 postcode dtype:", flat2['postcode'].dtype)
print("inc2016 postcode dtype:", inc2016['postcode'].dtype)
print()
print("flat2 sample postcodes:", [repr(x) for x in flat2['postcode'].head(3)])
print("inc2016 sample postcodes:", [repr(x) for x in inc2016['postcode'].head(3)])

flat2 postcode dtype: object
inc2016 postcode dtype: object

flat2 sample postcodes: ['4000', '4005', '4006']
inc2016 sample postcodes: ["'4000'", "'4005'", "'4006'"]


In [19]:
inc2016['postcode'] = inc2016['postcode'].str.replace("'", "", regex=False)
inc2021['postcode'] = inc2021['postcode'].str.replace("'", "", regex=False)

# verify they now match the rent format
print("inc2016 now:", [repr(x) for x in inc2016['postcode'].head(3)])
print("inc2021 now:", [repr(x) for x in inc2021['postcode'].head(3)])

inc2016 now: ["'4000'", "'4005'", "'4006'"]
inc2021 now: ["'4000'", "'4005'", "'4006'"]


In [20]:
sample = inc2016['postcode'].iloc[0]
print("The value:", repr(sample))
print("Length:", len(sample))
print("Each character and its code:")
for ch in sample:
    print(f"  {repr(ch)} -> {ord(ch)}")

The value: '4000'
Length: 4
Each character and its code:
  '4' -> 52
  '0' -> 48
  '0' -> 48
  '0' -> 48


In [22]:
sample = flat2['postcode'].iloc[0]
print("flat2 value:", repr(sample), "length:", len(sample))
for ch in sample:
    print(f"  {repr(ch)} -> {ord(ch)}")

print()
print("flat2 postcode lengths:", flat2['postcode'].str.len().unique())
print("inc2016 postcode lengths:", inc2016['postcode'].str.len().unique())

TypeError: object of type 'int' has no len()

In [23]:
# convert rent postcodes from number to text
flat2['postcode'] = flat2['postcode'].astype(str)

# verify the type and value
print("flat2 postcode now:", repr(flat2['postcode'].iloc[0]), "dtype:", flat2['postcode'].dtype)

# re-run the join
merged = flat2.merge(inc2016, on='postcode', how='inner').merge(inc2021, on='postcode', how='inner')
print("Postcodes after join:", merged.shape[0])
print(merged[['postcode', 'rent_2016', 'rent_2021', 'income_2016', 'income_2021']].head(10))

flat2 postcode now: '4000' dtype: object
Postcodes after join: 143
  postcode  rent_2016  rent_2021  income_2016  income_2021
0     4000      550.0      500.0         1728         1860
1     4005      492.0      542.0         2054         2258
2     4006      490.0      500.0         1618         1871
3     4007      430.0      485.0         1918         2130
4     4010      450.0      450.0         1724         1972
5     4011      340.0      370.0         1828         2115
6     4012      375.0      400.0         1726         2016
7     4013      320.0      307.0         1672         2041
8     4017      305.0      320.0         1591         1934
9     4019      275.0      330.0         1138         1283


In [24]:
merged['afford_2016'] = round(merged['rent_2016'] / merged['income_2016'] * 100, 1)
merged['afford_2021'] = round(merged['rent_2021'] / merged['income_2021'] * 100, 1)
merged['afford_change'] = round(merged['afford_2021'] - merged['afford_2016'], 1)

print(merged[['postcode', 'afford_2016', 'afford_2021', 'afford_change']].head(10))
print()
print("Median affordability 2016:", merged['afford_2016'].median(), "%")
print("Median affordability 2021:", merged['afford_2021'].median(), "%")
print("Worse (rent took bigger share):", (merged['afford_change'] > 0).sum())
print("Better:", (merged['afford_change'] < 0).sum())

  postcode  afford_2016  afford_2021  afford_change
0     4000         31.8         26.9           -4.9
1     4005         24.0         24.0            0.0
2     4006         30.3         26.7           -3.6
3     4007         22.4         22.8            0.4
4     4010         26.1         22.8           -3.3
5     4011         18.6         17.5           -1.1
6     4012         21.7         19.8           -1.9
7     4013         19.1         15.0           -4.1
8     4017         19.2         16.5           -2.7
9     4019         24.2         25.7            1.5

Median affordability 2016: 21.7 %
Median affordability 2021: 20.8 %
Worse (rent took bigger share): 63
Better: 77


In [25]:
merged.to_csv('affordability_flat2.csv', index=False)

from google.colab import files
files.download('affordability_flat2.csv')

print("Saved affordability_flat2.csv with", merged.shape[0], "postcodes")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved affordability_flat2.csv with 143 postcodes


In [26]:
bonds_raw = pd.read_excel('rta-bond-statistics.xlsx', sheet_name='2 pc-new-bonds', header=None)

bonds = bonds_raw.iloc[8:, [2, 3, 22, 42]].copy()
bonds.columns = ['postcode', 'dwelling', 'bonds_2016', 'bonds_2021']
bonds = bonds.dropna(subset=['postcode'])

# keep only Flat 2, match our analysis
bonds = bonds[bonds['dwelling'] == 'Flat 2'].copy()
bonds['postcode'] = bonds['postcode'].astype(str)

print("Flat 2 bond-count rows:", bonds.shape[0])
print(bonds.head(10))

Flat 2 bond-count rows: 319
   postcode dwelling bonds_2016 bonds_2021
9      4000   Flat 2        354        432
19     4005   Flat 2        310        244
29     4006   Flat 2        465        711
39     4007   Flat 2        205        208
49     4010   Flat 2         82         49
59     4011   Flat 2        134        100
69     4012   Flat 2        225        189
79     4013   Flat 2         35         23
89     4014   Flat 2         12          4
99     4017   Flat 2         20         15


In [27]:
merged = merged.merge(bonds[['postcode', 'bonds_2016', 'bonds_2021']], on='postcode', how='left')

# flag thin data: fewer than 10 bonds in either year
merged['thin_data'] = ((merged['bonds_2016'] < 10) | (merged['bonds_2021'] < 10)).map({True: 'Y', False: 'N'})

print("Postcodes after join:", merged.shape[0])
print("Thin-flagged:", (merged['thin_data'] == 'Y').sum(), "of", merged.shape[0])
print()
print(merged[['postcode', 'afford_change', 'bonds_2016', 'bonds_2021', 'thin_data']].head(12))

Postcodes after join: 143
Thin-flagged: 19 of 143

   postcode  afford_change bonds_2016 bonds_2021 thin_data
0      4000           -4.9        354        432         N
1      4005            0.0        310        244         N
2      4006           -3.6        465        711         N
3      4007            0.4        205        208         N
4      4010           -3.3         82         49         N
5      4011           -1.1        134        100         N
6      4012           -1.9        225        189         N
7      4013           -4.1         35         23         N
8      4017           -2.7         20         15         N
9      4019            1.5         64         45         N
10     4020           -2.0         58         52         N
11     4030           -1.9        241        214         N


In [28]:
merged.to_csv('affordability_flat2_flagged.csv', index=False)
from google.colab import files
files.download('affordability_flat2_flagged.csv')
print("Saved with", merged.shape[0], "postcodes,", (merged['thin_data']=='Y').sum(), "thin-flagged")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved with 143 postcodes, 19 thin-flagged


In [31]:
import urllib.request, zipfile, os

zip_url = "https://codeload.github.com/matthewproctor/australianpostcodes/zip/refs/heads/master"
req = urllib.request.Request(zip_url, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as r, open('postcodes_repo.zip', 'wb') as f:
    f.write(r.read())

with zipfile.ZipFile('postcodes_repo.zip') as z:
    z.extractall('postcodes_repo')

# find the csv inside the extracted repo
for folder, _, names in os.walk('postcodes_repo'):
    for n in names:
        if n == 'australian_postcodes.csv':
            path = os.path.join(folder, n)
            print("Found:", path)

geo = pd.read_csv(path, dtype={'postcode': str})
print("Total rows:", geo.shape[0])
print(geo[['postcode', 'locality', 'state', 'sa4name', 'lat', 'long']].head())

Found: postcodes_repo/australianpostcodes-master/australian_postcodes.csv
Total rows: 18559
  postcode                        locality state sa4name        lat       long
0     0200                             ANU   ACT     NaN -35.277700  149.11900
1     0200  Australian National University   ACT     NaN -35.277700  149.11890
2     0800                          DARWIN    NT  Darwin -12.458684  130.83668
3     0800                     DARWIN CITY    NT  Darwin -12.458684  130.83668
4     0801                          DARWIN    NT  Darwin -12.458684  130.83668


In [32]:
# keep QLD only, one row per postcode (first occurrence as representative)
qld_geo = geo[geo['state'] == 'QLD'].drop_duplicates(subset='postcode', keep='first')
qld_geo = qld_geo[['postcode', 'locality', 'sa4name', 'lat', 'long']].copy()
qld_geo.columns = ['postcode', 'locality', 'region', 'lat', 'long']

merged = merged.merge(qld_geo, on='postcode', how='left')

print("Postcodes after attaching geography:", merged.shape[0])
print(merged[['postcode', 'locality', 'region', 'afford_change']].head(10))

Postcodes after attaching geography: 143
  postcode       locality               region  afford_change
0     4000       BRISBANE  Brisbane Inner City           -4.9
1     4005       NEW FARM  Brisbane Inner City            0.0
2     4006   BOWEN BRIDGE  Brisbane Inner City           -3.6
3     4007          ASCOT  Brisbane Inner City            0.4
4     4010         ALBION  Brisbane Inner City           -3.3
5     4011      CLAYFIELD  Brisbane Inner City           -1.1
6     4012         NUNDAH  Brisbane Inner City           -1.9
7     4013      NORTHGATE     Brisbane - North           -4.1
8     4017  BRACKEN RIDGE     Brisbane - North           -2.7
9     4019       CLONTARF  Moreton Bay - South            1.5


In [33]:
checks = {
    '4218': 'Gold Coast',
    '4225': 'Gold Coast',
    '4000': 'Brisbane',
    '4566': 'Sunshine Coast',
    '4870': 'Cairns'
}

for pc, expected in checks.items():
    row = merged[merged['postcode'] == pc]
    if len(row):
        got = row['region'].values[0]
        ok = expected.split()[0] in str(got)
        print(f"{pc}: got '{got}', expected ~'{expected}'  {'OK' if ok else 'MISMATCH'}")
    else:
        print(f"{pc}: not in our 143")

4218: got 'Gold Coast', expected ~'Gold Coast'  OK
4225: got 'Gold Coast', expected ~'Gold Coast'  OK
4000: got 'Brisbane Inner City', expected ~'Brisbane'  OK
4566: got 'Sunshine Coast', expected ~'Sunshine Coast'  OK
4870: got 'Cairns', expected ~'Cairns'  OK


In [34]:
solid = merged[merged['thin_data'] == 'N']

worse = solid[solid['afford_change'] > 0]['region'].value_counts()
better = solid[solid['afford_change'] < 0]['region'].value_counts()

print("=== WORSENED (well-supported postcodes), by region ===")
print(worse.head(8))
print()
print("=== IMPROVED (well-supported postcodes), by region ===")
print(better.head(8))

=== WORSENED (well-supported postcodes), by region ===
region
Gold Coast             15
Sunshine Coast         10
Fitzroy                 5
Cairns                  5
Townsville              4
Wide Bay                3
Mackay                  3
Brisbane Inner City     2
Name: count, dtype: int64

=== IMPROVED (well-supported postcodes), by region ===
region
Brisbane Inner City    20
Brisbane - South        8
Moreton Bay - North     5
Ipswich                 5
Moreton Bay - South     5
Logan - Beaudesert      5
Brisbane - North        4
Brisbane - East         2
Name: count, dtype: int64


In [35]:
merged.to_csv('affordability_flat2_final.csv', index=False)
from google.colab import files
files.download('affordability_flat2_final.csv')
print("Final table saved:", merged.shape[0], "postcodes, fully analysed with geography")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Final table saved: 143 postcodes, fully analysed with geography


In [36]:
!pip install folium -q
import folium
print("folium ready")

folium ready


In [37]:
mappable = merged[(merged['thin_data'] == 'N') & merged['lat'].notna() & merged['long'].notna()].copy()

print("Mappable postcodes:", mappable.shape[0])
print("Latitude range:", round(mappable['lat'].min(), 1), "to", round(mappable['lat'].max(), 1))
print("Longitude range:", round(mappable['long'].min(), 1), "to", round(mappable['long'].max(), 1))
print()
print("Worse:", (mappable['afford_change'] > 0).sum(), " Better:", (mappable['afford_change'] < 0).sum())

Mappable postcodes: 124
Latitude range: -28.2 to -16.2
Longitude range: 145.3 to 153.5

Worse: 54  Better: 67


In [38]:
# centre roughly on QLD's populated coast
m = folium.Map(location=[-23.5, 147.0], zoom_start=6, tiles='CartoDB positron')

for _, row in mappable.iterrows():
    worsened = row['afford_change'] > 0
    colour = 'red' if worsened else 'blue'
    popup = (f"{row['postcode']} {row['locality']}<br>"
             f"{row['region']}<br>"
             f"Rent/income: {row['afford_2016']}% to {row['afford_2021']}%<br>"
             f"Change: {row['afford_change']} pts")
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=3 + min(abs(row['afford_change']), 9),
        color=colour,
        fill=True,
        fill_opacity=0.6,
        popup=folium.Popup(popup, max_width=250)
    ).add_to(m)

m  # display inline in Colab

In [39]:
m.save('qld_affordability_map.html')
from google.colab import files
files.download('qld_affordability_map.html')
print("Map saved as qld_affordability_map.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Map saved as qld_affordability_map.html
